# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print metadata fields
md = dataset.metadata
print('Title:', md.name)
print('Description:', md.description)
print('Version:', md.version)
print('License:', md.license)
print('Keywords:', getattr(md, 'keywords', None))
print('Published:', getattr(md, 'datePublished', None))

## 2. Data Overview
List available record sets (`@id`), their fields (`@id`), and sample columns.

In [ ]:
# List all available record sets and their fields

record_sets = dataset.metadata.recordSet
if isinstance(record_sets, dict):
    record_sets = [record_sets]
record_sets = record_sets or []  # fallback if None

print(f"Found {len(record_sets)} record set(s)")

for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', '<unknown>')}")
    print(f"  @id: {getattr(rs, '@id', None)}")
    fields = getattr(rs, 'field', [])
    if fields is not None and not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields ({len(fields)}):")
    for f in fields:
        print(f"    - {getattr(f, 'name', '<unnamed>')} (@id: {getattr(f, '@id', None)})  type: {getattr(f, 'dataType', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Load all tables by record set
import pprint

dataframes = {}
record_set_ids = []
record_set_names = []

for rs in dataset.metadata.recordSet or []:
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    record_set_names.append(getattr(rs, 'name', None))
    
    # Download records and load into a dataframe
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"---\nRecordSet {rs_id} ({getattr(rs, 'name', None)}):")
    if len(df.columns) > 0:
        print("Columns:", df.columns.tolist())
        print("Example records:\n", df.head())
    else:
        print("No data available.")
    print()

# Pick first available record set for further analysis
if len(record_set_ids) > 0:
    chosen_record_set = record_set_ids[0]  # Update below if you want to use another record set
    print("Working with RecordSet @id:", chosen_record_set)
else:
    chosen_record_set = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Below, we illustrate filtering and normalization for one numeric field.

In [ ]:
import numpy as np

if chosen_record_set is not None and chosen_record_set in dataframes and len(dataframes[chosen_record_set]) > 0:
    df = dataframes[chosen_record_set]

    # Try to auto-detect a numeric column to use for analysis
    # If the schema defines types, use that, otherwise autodetect
    numeric_field = None
    rs_obj = None
    for rs in dataset.metadata.recordSet or []:
        if getattr(rs, '@id', None) == chosen_record_set:
            rs_obj = rs
            break
    if rs_obj is not None:
        fields = getattr(rs_obj, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        # Find first numeric field
        for f in fields:
            if getattr(f, 'dataType', None) in ['schema:Number', 'schema:Float', 'schema:Integer']:
                # The field's @id may be used as the column name
                if getattr(f, '@id', None) in df.columns:
                    numeric_field = getattr(f, '@id', None)
                    break
        if numeric_field is None:
            # fallback: pick a numeric-type column
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
    
    if numeric_field is not None:
        print(f"Using numeric field (@id): {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean for demo filtering
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered rows where {numeric_field} > {threshold:.4f}: {len(filtered_df)} records")
        
        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() + 1e-6)
        print(filtered_df[[numeric_field, norm_col]].head())
        
        # Try grouping by a categorical/text column (preferably not the same as numeric)
        group_field = None
        for f in fields:
            # Omit numeric_field, look for fields of type schema:Text
            if getattr(f, '@id', None) != numeric_field and getattr(f, 'dataType', None) == 'schema:Text':
                if getattr(f, '@id', None) in df.columns:
                    group_field = getattr(f, '@id', None)
                    break
        if group_field:
            print(f"Grouping by field (@id): {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print("Group means:")
            print(grouped.head())
        else:
            print("No categorical field for grouping found.")
    else:
        print("No numeric field detected for analysis.")
else:
    print("No suitable record set or data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot a histogram of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set is not None and chosen_record_set in dataframes and len(dataframes[chosen_record_set]) > 0 and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(data=dataframes[chosen_record_set], x=numeric_field, kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library to load, inspect, and analyze a Croissant-defined FAIR dataset. We explored dataset structure, loaded tabular data by referencing record sets and fields via their `@id` values, performed simple EDA and visualizations, and demonstrated how to further prepare the data for machine learning or advanced statistical analysis.